# Clasificación, incertidumbre y explicabilidad con BloodMNIST

Este notebook recorre el mismo flujo que el Notebook 1: entrenamiento MONAI, discriminación y calibración, Monte Carlo Dropout, saliencia, Grad-CAM y sensibilidad por oclusión. Aquí la tarea es **multiclase de etiqueta única** (una de ocho categorías celulares); el Notebook 1 usa clasificación binaria de neumonía.

**Objetivos**
- Comparar discriminación y calibración con métricas multiclase y un reliability diagram top-label.
- Estimar variabilidad predictiva con Monte Carlo Dropout.
- Inspeccionar saliencia, Grad-CAM por clase y oclusión sin tratarlos como evidencia causal.

> **Uso educativo, no clínico.** BloodMNIST clasifica categorías de células sanguíneas; no es un sistema de diagnóstico hematológico.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

missing_packages = [
    package for package, module in (("monai", "monai"), ("medmnist", "medmnist"))
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import matplotlib.pyplot as plt
import numpy as np
import torch
from medmnist import BloodMNIST
from monai.data import DataLoader, Dataset
from monai.networks.nets import DenseNet121
from monai.transforms import Compose, EnsureChannelFirstd, EnsureTyped, ScaleIntensityd
from monai.utils import set_determinism
from monai.visualize import GradCAM

set_determinism(seed=23)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("MONAI:", __import__("monai").__version__)
print("MedMNIST:", __import__("medmnist").__version__)
print("Dispositivo:", device)

## 1. Clasificador MONAI

Entrenamos una DenseNet121 de MONAI para ocho categorías celulares. Con CUDA se usa el split completo de entrenamiento; en CPU se activa el submuestreo rápido, que puedes cambiar con `QUICK_RUN`. El objetivo es inspeccionar qué regiones contribuyen al score de clase; no es un sistema de diagnóstico hematológico.

In [ ]:
SEED = 23
QUICK_RUN = device.type != "cuda"
TRAIN_FRACTION = 0.05 if QUICK_RUN else 1.0
BATCH_SIZE = 16 if QUICK_RUN else 32
DATA_DIR = Path(os.environ.get("MONAI_DATA_DIRECTORY", Path.home() / ".cache" / "monai"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
set_determinism(seed=SEED)

train_raw = BloodMNIST(split="train", root=str(DATA_DIR), download=True, size=64, as_rgb=True)
valid_raw = BloodMNIST(split="val", root=str(DATA_DIR), download=True, size=64, as_rgb=True)
test_raw = BloodMNIST(split="test", root=str(DATA_DIR), download=True, size=64, as_rgb=True)
class_names = [
    "Basophil", "Eosinophil", "Erythroblast", "Immature granulocyte",
    "Lymphocyte", "Monocyte", "Neutrophil", "Platelet",
]
NUM_CLASSES = len(class_names)

def stratified_indices(labels, fraction, seed):
    labels = np.asarray(labels).reshape(-1)
    if fraction >= 1.0:
        return np.arange(len(labels))
    rng = np.random.default_rng(seed)
    selected = []
    for class_id in np.unique(labels):
        class_indices = np.flatnonzero(labels == class_id)
        count = max(1, int(round(len(class_indices) * fraction)))
        selected.extend(rng.choice(class_indices, count, replace=False).tolist())
    return np.asarray(sorted(selected), dtype=int)

def records_from_split(raw, indices=None):
    if indices is None:
        indices = np.arange(len(raw.imgs))
    labels = raw.labels.reshape(-1)
    return [{"image": raw.imgs[index], "label": int(labels[index])} for index in indices]

train_indices = stratified_indices(train_raw.labels, TRAIN_FRACTION, SEED)
train_records = records_from_split(train_raw, train_indices)
valid_records = records_from_split(valid_raw)
test_records = records_from_split(test_raw)
image_transform = Compose([
    EnsureChannelFirstd(keys="image", channel_dim=-1),
    ScaleIntensityd(keys="image"),
    EnsureTyped(keys="image", dtype=torch.float32),
    EnsureTyped(keys="label", dtype=torch.long),
])
train_ds = Dataset(data=train_records, transform=image_transform)
valid_ds = Dataset(data=valid_records, transform=image_transform)
test_ds = Dataset(data=test_records, transform=image_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

for split_name, raw, used_count in [
    ("train", train_raw, len(train_ds)),
    ("validation", valid_raw, len(valid_ds)),
    ("test", test_raw, len(test_ds)),
]:
    counts = np.bincount(raw.labels.reshape(-1).astype(int), minlength=NUM_CLASSES)
    print(f"{split_name}: usados={used_count}/{len(raw.imgs)} | por clase={counts.tolist()}")
print(f"QUICK_RUN={QUICK_RUN} | train fraction={TRAIN_FRACTION} | batch size={BATCH_SIZE}")

fig, axes = plt.subplots(2, 4, figsize=(11, 6))
for class_id, ax in enumerate(axes.flat):
    sample_index = next(i for i, record in enumerate(train_records) if record["label"] == class_id)
    sample = train_ds[sample_index]["image"].permute(1, 2, 0).numpy()
    ax.imshow(sample)
    ax.set_title(class_names[class_id])
    ax.axis("off")
plt.tight_layout(); plt.show()
print("Tamaño de entrada MONAI:", tuple(train_ds[0]["image"].shape))

In [ ]:
model = DenseNet121(
    spatial_dims=2,
    in_channels=3,
    out_channels=NUM_CLASSES,
    init_features=16,
    dropout_prob=0.1,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()
NUM_EPOCHS = 2 if QUICK_RUN else 8

def evaluate_accuracy(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            correct += int((model(images).argmax(dim=1) == labels).sum().item())
            total += len(labels)
    return correct / max(total, 1)

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(
        f"Época {epoch + 1}/{NUM_EPOCHS}: "
        f"train loss={running_loss / len(train_loader):.4f} | "
        f"validation accuracy={evaluate_accuracy(valid_loader):.3f}"
    )

print(f"Test accuracy: {evaluate_accuracy(test_loader):.3f}")
example_batch = next(iter(test_loader))
example_images = example_batch["image"].to(device)
example_labels = example_batch["label"].to(device)
with torch.no_grad():
    example_logits = model(example_images)
correct_indices = torch.nonzero(
    example_logits.argmax(dim=1) == example_labels, as_tuple=False
).flatten()
example_index = int(correct_indices[0]) if len(correct_indices) else 0
image = example_images[example_index : example_index + 1]
true_class = int(example_labels[example_index].item())
with torch.no_grad():
    probabilities = model(image).softmax(dim=1)[0].cpu().numpy()
predicted_class = int(probabilities.argmax())
print(
    f"Ejemplo test | real: {class_names[true_class]} | "
    f"predicción: {class_names[predicted_class]} ({probabilities[predicted_class]:.2f})"
)

## 2. Discriminación, calibración e incertidumbre

Como en el notebook binario, evaluamos separación entre clases, fiabilidad de probabilidades y variabilidad entre pasadas. BloodMNIST es **multiclase de etiqueta única**: cada imagen pertenece a una sola de ocho categorías. Calculamos Brier multiclase, ECE top-label y MC Dropout.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score


def expected_calibration_error(correct, confidence, n_bins=10):
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for bin_index, (lower, upper) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
        if bin_index == n_bins - 1:
            selected = (confidence >= lower) & (confidence <= upper)
        else:
            selected = (confidence >= lower) & (confidence < upper)
        if selected.any():
            ece += selected.mean() * abs(correct[selected].mean() - confidence[selected].mean())
    return ece


def predict_probabilities(loader):
    model.eval()
    probabilities, labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch["image"].to(device))
            probabilities.append(logits.softmax(dim=1).cpu())
            labels.append(batch["label"].cpu())
    return torch.cat(probabilities).numpy(), torch.cat(labels).numpy()


valid_probabilities, valid_targets = predict_probabilities(valid_loader)
valid_predictions = valid_probabilities.argmax(axis=1)
valid_confidence = valid_probabilities.max(axis=1)
valid_correct = (valid_predictions == valid_targets).astype(float)
valid_one_hot = np.eye(NUM_CLASSES)[valid_targets]
multiclass_brier = np.mean(np.sum((valid_probabilities - valid_one_hot) ** 2, axis=1))
top_label_ece = expected_calibration_error(valid_correct, valid_confidence)
print(f"Validación | Brier multiclase: {multiclass_brier:.4f} | ECE top-label: {top_label_ece:.4f}")

bin_edges = np.linspace(0.0, 1.0, 11)
mean_confidence, observed_accuracy = [], []
for bin_index, (lower, upper) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
    selected = ((valid_confidence >= lower) & (valid_confidence <= upper)) if bin_index == 9 else ((valid_confidence >= lower) & (valid_confidence < upper))
    if selected.any():
        mean_confidence.append(valid_confidence[selected].mean())
        observed_accuracy.append(valid_correct[selected].mean())
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Calibración perfecta")
ax.plot(mean_confidence, observed_accuracy, "o-", label="BloodMNIST validation")
ax.set_xlabel("Confianza de la clase más probable")
ax.set_ylabel("Exactitud observada")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.2); ax.legend()
plt.tight_layout(); plt.show()

test_probabilities, test_targets = predict_probabilities(test_loader)
test_predictions = test_probabilities.argmax(axis=1)
print(f"Test oficial | balanced accuracy: {balanced_accuracy_score(test_targets, test_predictions):.4f}")
print(f"Test oficial | macro F1: {f1_score(test_targets, test_predictions, average='macro'):.4f}")


def mc_dropout_predictions(model, images, n_passes=20):
    model.eval()
    dropout_types = (torch.nn.Dropout, torch.nn.Dropout2d, torch.nn.Dropout3d)
    for module in model.modules():
        if isinstance(module, dropout_types):
            module.train()
    with torch.no_grad():
        samples = torch.stack([model(images).softmax(dim=1) for _ in range(n_passes)])
    model.eval()
    return samples


mc_count = 8 if QUICK_RUN else 24
mc_images = next(iter(test_loader))["image"][:mc_count].to(device)
mc_samples = mc_dropout_predictions(model, mc_images)
mc_uncertainty = mc_samples.std(dim=0).amax(dim=1).cpu().numpy()
print(f"Dispersión media top-class en {mc_count} casos: {mc_uncertainty.mean():.4f}")
plt.hist(mc_uncertainty, bins=8)
plt.xlabel("Desviación estándar MC de la clase más probable")
plt.ylabel("Número de imágenes")
plt.title("Variabilidad predictiva en BloodMNIST")
plt.grid(alpha=0.2); plt.show()

## 3. Saliencia: gradiente respecto de la imagen

El gradiente local indica sensibilidad del score de clase ante pequeñas perturbaciones de los píxeles. Visualizamos su magnitud; no debe interpretarse como causalidad, atención consciente ni relevancia clínica.

In [ ]:
saliency_input = image.detach().clone().requires_grad_(True)
model.zero_grad(set_to_none=True)
score = model(saliency_input)[0, predicted_class]
score.backward()
saliency = saliency_input.grad.detach().abs().amax(dim=1)[0].cpu().numpy()
image_rgb = image[0].detach().cpu().permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(image_rgb)
axes[0].set_title("Imagen")
axes[1].imshow(saliency[0], cmap="inferno")
axes[1].set_title("Saliencia |gradiente|")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Grad-CAM para la clase elegida

Grad-CAM combina mapas de características de una capa convolucional con gradientes de la clase objetivo. En BloodMNIST seleccionamos la clase predicha por la DenseNet; se puede cambiar `predicted_class` por cualquier índice de 0 a 7 para explicar otra categoría. El mapa no demuestra causalidad.

In [ ]:
gradcam = GradCAM(
    nn_module=model,
    target_layers="features.denseblock4",
    postprocessing=lambda activation: activation,
)
cam = gradcam(image, class_idx=predicted_class)
cam_map = cam[0, 0].detach().cpu().numpy()
cam_map = (cam_map - cam_map.min()) / (cam_map.max() - cam_map.min() + 1e-8)

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image_rgb)
ax.imshow(cam_map, cmap="jet", alpha=0.42)
ax.set_title(f"Grad-CAM | clase {class_names[predicted_class]}")
ax.axis("off"); plt.show()

## 5. Sensibilidad por oclusión

Reemplazamos ventanas de 16×16 por cero y medimos cuánto cae el score de la clase predicha. En modo rápido son 16 inferencias por imagen de 64×64. El resultado depende del tamaño y valor del parche; no es una prueba causal.

In [ ]:
def occlusion_map(model, image, class_index, patch_size=16, stride=16):
    model.eval()
    height, width = image.shape[-2:]
    score_map = np.zeros((height, width), dtype=np.float32)
    counts = np.zeros_like(score_map)
    with torch.no_grad():
        baseline = model(image).softmax(dim=1)[0, class_index].item()
        for top in range(0, height - patch_size + 1, stride):
            for left in range(0, width - patch_size + 1, stride):
                occluded = image.clone()
                occluded[:, :, top:top + patch_size, left:left + patch_size] = 0
                score_drop = baseline - model(occluded).softmax(dim=1)[0, class_index].item()
                score_map[top:top + patch_size, left:left + patch_size] += score_drop
                counts[top:top + patch_size, left:left + patch_size] += 1
    return score_map / np.maximum(counts, 1)

occlusion = occlusion_map(model, image, predicted_class)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image_rgb)
ax.imshow(occlusion, cmap="coolwarm", alpha=0.5)
ax.set_title("Caída del score al ocluir")
ax.axis("off"); plt.show()

## Comparación y discusión

1. ¿Qué diferencias observas entre Brier/ECE top-label multiclase y las métricas binarias del Notebook 1?
2. ¿La dispersión de MC Dropout cambia según el ejemplo? ¿Qué no demuestra esa señal?
3. ¿Qué regiones resaltan saliencia y Grad-CAM para distintas categorías? ¿Coinciden con oclusión?
4. ¿Por qué estos mapas no son máscaras de segmentación ni explicaciones causales?

**Idea clave:** ambos notebooks comparten el mismo flujo; cambian el número y la semántica de las clases. Discriminación, calibración, incertidumbre y explicabilidad siguen respondiendo preguntas distintas.